In [1]:
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

In [2]:
data_url = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "course_lead_scoring_2026.csv"
)

df = pd.read_csv(data_url)

print(df.shape)
display(df.head())
df.info()

(5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [3]:
target = "converted"

categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_columns = [
    column
    for column in df.select_dtypes(include="number").columns
    if column != target
]

/tmp/ipykernel_3793/3971155521.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


In [4]:
df[categorical_columns] = df[categorical_columns].fillna("NA")
df[numerical_columns] = df[numerical_columns].fillna(0.0)

In [5]:
df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
)

df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=42,
)

In [6]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.pop(target).values
y_val = df_val.pop(target).values
y_test = df_test.pop(target).values

In [7]:
vectorizer = DictVectorizer(sparse=False)

X_train = vectorizer.fit_transform(df_train.to_dict(orient="records"))
X_val = vectorizer.transform(df_val.to_dict(orient="records"))

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42,
)

model.fit(X_train, y_train)

predictions = model.predict(X_val)
validation_accuracy = accuracy_score(y_val, predictions)

In [9]:
industry_counts = df["industry"].value_counts()

print(industry_counts)

most_frequent_industry = df["industry"].mode().iloc[0]
print("\nMost frequent value:", most_frequent_industry)

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

Most frequent value: technology


In [10]:
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

for first_feature, second_feature in pairs:
    correlation = df[first_feature].corr(df[second_feature])

    print(
        f"{first_feature} and {second_feature}: "
        f"{correlation:.4f}"
    )

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228


In [11]:
from sklearn.metrics import mutual_info_score

features = [
    "industry",
    "location",
    "lead_source",
    "employment_status",
]

mi_scores = {}

for feature in features:
    score = mutual_info_score(
        df_train[feature],
        y_train,
    )

    mi_scores[feature] = round(score, 2)

print(mi_scores)

{'industry': 0.0, 'location': 0.0, 'lead_source': 0.03, 'employment_status': 0.02}


In [13]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

vectorizer = DictVectorizer(sparse=False)

X_train = vectorizer.fit_transform(
    df_train.to_dict(orient="records")
)

X_val = vectorizer.transform(
    df_val.to_dict(orient="records")
)

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42,
)

model.fit(X_train, y_train)

predictions = model.predict(X_val)
accuracy = accuracy_score(y_val, predictions)

print("Exact accuracy:", accuracy)
print("Rounded accuracy:", round(accuracy, 2))

Exact accuracy: 0.645
Rounded accuracy: 0.65


In [17]:
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


def train_and_measure_accuracy(
    training_dataframe,
    validation_dataframe,
    y_training,
    y_validation,
    c_value=1.0,
):
    vectorizer = DictVectorizer(sparse=False)

    X_training = vectorizer.fit_transform(
        training_dataframe.to_dict(orient="records")
    )

    X_validation = vectorizer.transform(
        validation_dataframe.to_dict(orient="records")
    )

    model = LogisticRegression(
        solver="liblinear",
        C=c_value,
        max_iter=1000,
        random_state=42,
    )

    model.fit(X_training, y_training)
    predictions = model.predict(X_validation)

    return accuracy_score(y_validation, predictions)


candidate_features = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count",
]

original_accuracy = train_and_measure_accuracy(
    df_train,
    df_val,
    y_train,
    y_val,
)

results = []

for feature in candidate_features:
    reduced_train = df_train.drop(columns=[feature])
    reduced_val = df_val.drop(columns=[feature])

    accuracy_without_feature = train_and_measure_accuracy(
        reduced_train,
        reduced_val,
        y_train,
        y_val,
    )

    difference = original_accuracy - accuracy_without_feature

    results.append({
        "removed_feature": feature,
        "original_accuracy": original_accuracy,
        "accuracy_without_feature": accuracy_without_feature,
        "difference": difference,
    })

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        by="difference",
        ascending=True,
    )
)

,removed_feature,original_accuracy,accuracy_without_feature,difference
1,number_of_courses_viewed,0.645,0.643,0.002
0,lead_source,0.645,0.642,0.003
2,interaction_count,0.645,0.601,0.044


In [18]:
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# Convert categorical features into numerical features
vectorizer = DictVectorizer(sparse=False)

X_train = vectorizer.fit_transform(
    df_train.to_dict(orient="records")
)

X_val = vectorizer.transform(
    df_val.to_dict(orient="records")
)


# Values provided in Question 6
c_values = [
    0.000001,
    0.00001,
    0.0001,
    0.001,
]

results = []

for c_value in c_values:
    model = LogisticRegression(
        solver="liblinear",
        C=c_value,
        max_iter=1000,
        random_state=42,
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)
    accuracy = accuracy_score(y_val, predictions)

    results.append({
        "C": c_value,
        "exact_accuracy": accuracy,
        "rounded_accuracy": round(accuracy, 3),
    })


results_df = pd.DataFrame(results)

# Highest accuracy first; smallest C first when accuracies tie
results_df = results_df.sort_values(
    by=["rounded_accuracy", "C"],
    ascending=[False, True],
)

display(results_df)

,C,exact_accuracy,rounded_accuracy
3,0.001000,0.645,0.645
2,0.000100,0.613,0.613
0,0.000001,0.598,0.598
1,0.000010,0.598,0.598
